In [1]:
%pip install python-dotenv openai datasets math_verify tqdm torch aiolimiter

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
import os
from openai import AsyncOpenAI
from math_verify import parse

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

client = AsyncOpenAI(
	base_url="https://integrate.api.nvidia.com/v1",
	api_key=NVIDIA_API_KEY,
	max_retries=0,
)

prompt = "What is 13 times 17? Box your answer."
gold = "221"

completion = await client.chat.completions.create(
	model="nvidia/nemotron-3.5-lightning-30b-a3b",
	messages=[{"role": "user", "content": prompt}],
	max_tokens=2**10,
    n=8,
	extra_body={
        "chat_template_kwargs": {"enable_thinking": True},
        "reasoning_budget": 2**10
    },
)

print("*"*20, "Prompt", "*"*20)
print(prompt)

parsed_gold = parse(gold) or [None]
for i, choice in enumerate(completion.choices):
  message = choice.message
  parsed_answer = parse(message.content) or [None]
  correct = parsed_gold[0] == parsed_answer[0]

  print("*"*20, f"Generation {i+1}: {parsed_answer[0]} ({'correct' if correct else 'incorrect'})", "*"*20)
  if correct:
    print(message.reasoning_content)

******************** Prompt ********************
What is 13 times 17? Box your answer.
******************** Generation 1: 221 (correct) ********************
Here's a thinking process:

1.  **Identify the user's request**: The user wants to know the product of 13 and 17, and they want the answer boxed.

2.  **Calculate 13 × 17**:
   - I can compute this mentally or step-by-step.
   - 13 × 17 = 13 × (10 + 7) = 130 + 91 = 221
   - Or: 17 × 10 = 170, 17 × 3 = 51, 170 + 51 = 221
   - Or: 13 × 17 = (15 - 2)(15 + 2) = 15² - 2² = 225 - 4 = 221
   - All methods give 221.

3.  **Format the output**: The user said "Box your answer." I should present the final answer clearly, likely using LaTeX \boxed{221} or just plain text with boxes. Since it's a math context, \boxed{221} is standard. I'll output exactly that.

4.  **Construct the response**: 
   - Calculate it.
   - Box the result.
   - Keep it concise.

   Output: \boxed{221}
   Or maybe: The answer is \boxed{221}. I'll just give the
********

In [4]:
from datasets import load_dataset

ds = load_dataset("open-r1/OpenR1-Math-220k", "default")
ds

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['problem', 'solution', 'answer', 'problem_type', 'question_type', 'source', 'uuid', 'is_reasoning_complete', 'generations', 'correctness_math_verify', 'correctness_llama', 'finish_reasons', 'correctness_count', 'messages'],
        num_rows: 93733
    })
})

In [ ]:
from aiolimiter import AsyncLimiter
import logging
from tqdm.contrib.logging import logging_redirect_tqdm
from tqdm.asyncio import tqdm_asyncio
from math_verify import verify
import torch
from datasets import Dataset

logging.basicConfig(level=logging.INFO)

limiter = AsyncLimiter(40)
async def create_completion(*args, **kwargs):
	while True:
		try:
			async with limiter:
				return await client.chat.completions.create(*args, **kwargs)
		except:
			pass

async def generate_dataset(prompts, golds, **kwargs):
	with logging_redirect_tqdm():
		dataset_dict = {
			"prompt": [],
			"outputs": [],
			"advantages": [],
		}
		futures = []
		for prompt in prompts:
			futures.append(create_completion(
				**kwargs,
				messages=[{"role": "user", "content": prompt}],
			))
		completions = await tqdm_asyncio.gather(*futures, desc="Creating completions")
		for prompt, completion, gold in zip(prompts, completions, golds):
			gold = parse(gold)

			outputs = []
			rewards = []
			for choice in completion.choices:
				message = choice.message
				answer = parse(message.content)
				correct = verify(gold, answer)

				outputs.append(f"<think>{message.reasoning_content}</think>{message.content}")
				rewards.append(1.0 if correct else 0.0)
			rewards = torch.tensor(rewards)

			rewards_std = rewards.std()
			if rewards_std < 1e-5:
				advantages = torch.zeros_like(rewards)
			else:
				advantages = (rewards - rewards.mean()) / rewards_std

			dataset_dict["prompt"].append(prompt)
			dataset_dict["outputs"].append(outputs)
			dataset_dict["advantages"].append(advantages.tolist())
		return Dataset.from_dict(dataset_dict)

example_ds = await generate_dataset(
	prompts=[
		"Pick an random integer from 1 to 3. Don't pick 2. Box your answer.",
		"What is 8 times 3? Box your answer.",
	],
	golds=["3", "24"],
	model="nvidia/nemotron-3.5-lightning-30b-a3b",
	max_tokens=2**10,
    n=8,
	extra_body={
        "chat_template_kwargs": {"enable_thinking": True},
        "reasoning_budget": 2**10,
    },
)
example_ds[:]

INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/

CancelledError: 

INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"


In [ ]:
input_ds = ds["train"].shuffle().select(range(128))
output_ds = await generate_dataset(
	prompts=input_ds["problem"],
	golds=input_ds["answer"],
	model="nvidia/nemotron-3.5-lightning-30b-a3b",
	max_tokens=2**15,
    n=64,
	extra_body={
        "chat_template_kwargs": {"enable_thinking": True},
        "reasoning_budget": 2**15,
    },
)
output_ds

INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/